In [1]:
import pandas as pd

# Load your crop dataset
cy = pd.read_csv("Crop_Production_Statistics.csv")
cy

,State,District,Crop,Crop_Year,Season,Area,Production,Yield
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75
...,...,...,...,...,...,...,...,...
345331,West Bengal,PURULIA,Wheat,2015,Rabi,855.0,1241.0,1.45
345332,West Bengal,PURULIA,Wheat,2016,Rabi,1366.0,2415.0,1.77
345333,West Bengal,PURULIA,Wheat,2017,Rabi,1052.0,2145.0,2.04
345334,West Bengal,PURULIA,Wheat,2018,Rabi,833.0,2114.0,2.54


In [2]:
import requests
import pandas as pd

lat = 12.97
lon = 77.59
# NASA POWER API endpoint for monthly climate data
url = "https://power.larc.nasa.gov/api/temporal/monthly/point"

params = {
    "parameters": "T2M",
    "community": "AG",
    "longitude": lon,
    "latitude": lat,
    "start": 1997,
    "end": 2020,
    "format": "JSON"
}

# Send request to NASA POWER API
response = requests.get(url, params=params)

#Check whether the request was successful
print("Status Code:", response.status_code)

# Convert JSON response to Python dictionary
data_json = response.json()

data = data_json['properties']['parameter']['T2M']

df = pd.DataFrame(list(data.items()), columns=['date', 'temp'])

# Extract year
df['year'] = df['date'].str[:4].astype(int)

# Convert to yearly average temperature
temp_yearly = df.groupby('year')['temp'].mean().reset_index()

temp_yearly.rename(columns={
    'year': 'crop_year',
    'temp': 'temperature'
}, inplace=True)

print(temp_yearly.head())

Status Code: 200
   crop_year  temperature
0       1997    23.542308
1       1998    23.737692
2       1999    22.943846
3       2000    23.218462
4       2001    23.382308


In [3]:
cy.isnull().sum()

State            0
District         0
Crop             9
Crop_Year        0
Season           0
Area             0
Production    4948
Yield            0
dtype: int64

In [4]:
cy.columns = cy.columns.str.strip().str.lower()

In [5]:
# Merge the crop dataset (cy) with the yearly temperature dataset
df_crop = cy.merge(
    temp_yearly[['crop_year','temperature']],
    on='crop_year',
    how='left'
)

In [6]:
df_crop

,state,district,crop,crop_year,season,area,production,yield,temperature
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40,23.250000
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40,23.250000
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74,22.756923
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64,22.756923
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75,23.503077
...,...,...,...,...,...,...,...,...,...
345331,West Bengal,PURULIA,Wheat,2015,Rabi,855.0,1241.0,1.45,23.865385
345332,West Bengal,PURULIA,Wheat,2016,Rabi,1366.0,2415.0,1.77,24.496154
345333,West Bengal,PURULIA,Wheat,2017,Rabi,1052.0,2145.0,2.04,23.740000
345334,West Bengal,PURULIA,Wheat,2018,Rabi,833.0,2114.0,2.54,23.967692


In [7]:
import pandas as pd
import glob
import os

In [8]:
import glob
# Path containing all IMD weather files
folder = "C:/Users/lohit/Downloads/IMD_MetData"

files = glob.glob(folder + "/data_*")

print(len(files))  # check if files are detected

435


In [9]:
import pandas as pd
import os

all_data = []

for file in files:
    df = pd.read_csv(file, sep="\s+", header=None)
    
    filename = os.path.basename(file)
    parts = filename.split("_")
    
    lat = float(parts[1])
    lon = float(os.path.splitext(parts[2])[0])
    
    df['lat'] = lat
    df['lon'] = lon
    
    all_data.append(df)

df_weather = pd.concat(all_data, ignore_index=True)

In [10]:
print(df_weather.head())
print(df_weather[['lat','lon']].drop_duplicates().head())

     0      1      2     lat   lon
0  0.0  29.86  19.93  10.125  76.0
1  0.0  29.67  19.36  10.125  76.0
2  0.0  29.75  19.37  10.125  76.0
3  0.0  29.81  18.90  10.125  76.0
4  0.0  29.51  18.87  10.125  76.0
           lat   lon
0       10.125  76.0
77799   10.125  77.0
181531  10.125  78.0
285263  10.375  76.0
363062  10.375  77.0


In [11]:
df_weather.columns = ['rainfall', 'tmax', 'tmin', 'lat', 'lon']

In [12]:
df.head()

,0,1,2,lat,lon
0,0.0,30.16,17.76,15.625,79.0
1,0.0,29.35,16.87,15.625,79.0
2,0.0,29.01,15.73,15.625,79.0
3,0.0,28.92,16.40,15.625,79.0
4,0.0,29.55,17.06,15.625,79.0


In [13]:
df.columns

Index([0, 1, 2, 'lat', 'lon'], dtype='object')

In [14]:
df_weather.head()

,rainfall,tmax,tmin,lat,lon
0,0.0,29.86,19.93,10.125,76.0
1,0.0,29.67,19.36,10.125,76.0
2,0.0,29.75,19.37,10.125,76.0
3,0.0,29.81,18.90,10.125,76.0
4,0.0,29.51,18.87,10.125,76.0


In [16]:
import pandas as pd
# Extract month from date column

dates = pd.date_range(start="1951-01-01", end="2021-12-31")

df_weather['date'] = dates.repeat(len(df_weather) // len(dates))

In [17]:
print(df_weather.head())
print(df_weather.tail())

   rainfall   tmax   tmin     lat   lon       date
0       0.0  29.86  19.93  10.125  76.0 1951-01-01
1       0.0  29.67  19.36  10.125  76.0 1951-01-01
2       0.0  29.75  19.37  10.125  76.0 1951-01-01
3       0.0  29.81  18.90  10.125  76.0 1951-01-01
4       0.0  29.51  18.87  10.125  76.0 1951-01-01
          rainfall   tmax   tmin     lat   lon       date
11280850       0.0  31.03  19.41  15.625  79.0 2021-12-31
11280851       0.0  31.10  19.32  15.625  79.0 2021-12-31
11280852       0.0  31.41  19.61  15.625  79.0 2021-12-31
11280853       0.0  31.09  20.01  15.625  79.0 2021-12-31
11280854       0.0  30.43  20.97  15.625  79.0 2021-12-31


In [18]:
df_weather['year'] = df_weather['date'].dt.year
df_weather['month'] = df_weather['date'].dt.month

In [19]:
kharif = df_weather[df_weather['month'].isin([6,7,8,9])]

In [20]:
df_kharif = kharif.groupby(['lat','lon','year']).agg({
    'rainfall':'sum'
}).reset_index()

In [21]:
import pandas as pd
# Read district location dataset
df_loc = pd.read_csv("district_state_lon_lat.csv")


In [22]:
df_loc.columns

Index(['city', 'lat', 'lon', 'state'], dtype='object')

In [24]:
df_crop.columns

Index(['state', 'district', 'crop', 'crop_year', 'season', 'area',
       'production', 'yield', 'temperature'],
      dtype='object')

In [25]:
df_loc = cy[['district','state']].drop_duplicates()

In [26]:
# Extract all unique weather grid points
grid_points = df_weather[['lat','lon']].drop_duplicates().reset_index(drop=True)

In [27]:
import numpy as np

# Repeat grid points to match df_loc size
repeat_factor = int(np.ceil(len(df_loc) / len(grid_points)))

grid_extended = pd.concat([grid_points]*repeat_factor, ignore_index=True)

# Now assign safely
df_loc['lat'] = grid_extended['lat'][:len(df_loc)].values
df_loc['lon'] = grid_extended['lon'][:len(df_loc)].values

In [28]:
df_loc.columns

Index(['district', 'state', 'lat', 'lon'], dtype='object')

In [29]:
df_crop.columns = df_crop.columns.str.strip().str.lower()

In [30]:
df_crop

,state,district,crop,crop_year,season,area,production,yield,temperature
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40,23.250000
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40,23.250000
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74,22.756923
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64,22.756923
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75,23.503077
...,...,...,...,...,...,...,...,...,...
345331,West Bengal,PURULIA,Wheat,2015,Rabi,855.0,1241.0,1.45,23.865385
345332,West Bengal,PURULIA,Wheat,2016,Rabi,1366.0,2415.0,1.77,24.496154
345333,West Bengal,PURULIA,Wheat,2017,Rabi,1052.0,2145.0,2.04,23.740000
345334,West Bengal,PURULIA,Wheat,2018,Rabi,833.0,2114.0,2.54,23.967692


In [31]:
df_loc.columns

Index(['district', 'state', 'lat', 'lon'], dtype='object')

In [32]:
import numpy as np

# Repeat grid points to match df_loc size
repeat_factor = int(np.ceil(len(df_loc) / len(grid_points)))

grid_extended = pd.concat([grid_points]*repeat_factor, ignore_index=True)

# Now assign safely
df_loc['lat'] = grid_extended['lat'][:len(df_loc)].values
df_loc['lon'] = grid_extended['lon'][:len(df_loc)].values

In [33]:
df_crop.columns = df_crop.columns.str.strip().str.lower()
df_crop['state'] = df_crop['state'].str.upper().str.strip()
df_crop['district'] = df_crop['district'].str.upper().str.strip()

In [35]:
print(df_loc.columns)

Index(['district', 'state', 'lat', 'lon'], dtype='object')


In [36]:
# Calculate average rainfall for each year
rain_year = df_kharif.groupby('year')['rainfall'].mean().reset_index()

In [37]:
# Merge yearly rainfall data with the crop dataset 
df_final = df_crop.merge(
    rain_year,
    left_on='crop_year',
    right_on='year',
    how='left'
)

In [38]:
df_final

,state,district,crop,crop_year,season,area,production,yield,temperature,year,rainfall
0,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40,23.250000,2007,110072.570
1,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40,23.250000,2007,110072.570
2,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74,22.756923,2008,40556.560
3,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64,22.756923,2008,40556.560
4,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75,23.503077,2009,106089.450
...,...,...,...,...,...,...,...,...,...,...,...
345331,WEST BENGAL,PURULIA,Wheat,2015,Rabi,855.0,1241.0,1.45,23.865385,2015,93275.670
345332,WEST BENGAL,PURULIA,Wheat,2016,Rabi,1366.0,2415.0,1.77,24.496154,2016,45879.095
345333,WEST BENGAL,PURULIA,Wheat,2017,Rabi,1052.0,2145.0,2.04,23.740000,2017,103574.290
345334,WEST BENGAL,PURULIA,Wheat,2018,Rabi,833.0,2114.0,2.54,23.967692,2018,206875.515


In [39]:
# Fill missing rainfall values with the overall average rainfall
df_final['rainfall'] = df_final['rainfall'].fillna(df_final['rainfall'].mean())

In [40]:
total_values = df_final.size
non_null = df_final.count().sum()
null_values = df_final.isnull().sum().sum()

print("Total values:", total_values)
print("Non-null values:", non_null)
print("Null values:", null_values)
print("Missing %:", (null_values / total_values) * 100)

Total values: 3798696
Non-null values: 3793739
Null values: 4957
Missing %: 0.13049214783178228


In [41]:
# Fill remaining numeric missing values
df_final = df_final.fillna(df_final.median(numeric_only=True))

In [42]:
df_final.isnull().sum().sum()

np.int64(9)

In [43]:
df_final = df_final.dropna()

In [44]:
df_final.isnull().sum().sum()

np.int64(0)

In [46]:
df_final

,state,district,crop,crop_year,season,area,production,yield,temperature,year,rainfall
0,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40,23.250000,2007,110072.570
1,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40,23.250000,2007,110072.570
2,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74,22.756923,2008,40556.560
3,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64,22.756923,2008,40556.560
4,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75,23.503077,2009,106089.450
...,...,...,...,...,...,...,...,...,...,...,...
345331,WEST BENGAL,PURULIA,Wheat,2015,Rabi,855.0,1241.0,1.45,23.865385,2015,93275.670
345332,WEST BENGAL,PURULIA,Wheat,2016,Rabi,1366.0,2415.0,1.77,24.496154,2016,45879.095
345333,WEST BENGAL,PURULIA,Wheat,2017,Rabi,1052.0,2145.0,2.04,23.740000,2017,103574.290
345334,WEST BENGAL,PURULIA,Wheat,2018,Rabi,833.0,2114.0,2.54,23.967692,2018,206875.515


In [47]:
# Remove the 'year' column
df_final = df_final.drop(columns=['year'])

In [48]:
df_final

,state,district,crop,crop_year,season,area,production,yield,temperature,rainfall
0,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40,23.250000,110072.570
1,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40,23.250000,110072.570
2,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74,22.756923,40556.560
3,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64,22.756923,40556.560
4,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75,23.503077,106089.450
...,...,...,...,...,...,...,...,...,...,...
345331,WEST BENGAL,PURULIA,Wheat,2015,Rabi,855.0,1241.0,1.45,23.865385,93275.670
345332,WEST BENGAL,PURULIA,Wheat,2016,Rabi,1366.0,2415.0,1.77,24.496154,45879.095
345333,WEST BENGAL,PURULIA,Wheat,2017,Rabi,1052.0,2145.0,2.04,23.740000,103574.290
345334,WEST BENGAL,PURULIA,Wheat,2018,Rabi,833.0,2114.0,2.54,23.967692,206875.515


In [49]:
# Convert rainfall units by dividing by 100
df_final['rainfall'] = df_final['rainfall'] / 100

In [50]:
df_final

,state,district,crop,crop_year,season,area,production,yield,temperature,rainfall
0,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40,23.250000,1100.72570
1,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40,23.250000,1100.72570
2,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74,22.756923,405.56560
3,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64,22.756923,405.56560
4,ANDAMAN AND NICOBAR ISLAND,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75,23.503077,1060.89450
...,...,...,...,...,...,...,...,...,...,...
345331,WEST BENGAL,PURULIA,Wheat,2015,Rabi,855.0,1241.0,1.45,23.865385,932.75670
345332,WEST BENGAL,PURULIA,Wheat,2016,Rabi,1366.0,2415.0,1.77,24.496154,458.79095
345333,WEST BENGAL,PURULIA,Wheat,2017,Rabi,1052.0,2145.0,2.04,23.740000,1035.74290
345334,WEST BENGAL,PURULIA,Wheat,2018,Rabi,833.0,2114.0,2.54,23.967692,2068.75515
